# 🔥 Mossad-pentast-ai - Cloud Model Fusion & Standalone Weight Merger

Fuses all trained LoRA security adapters (27,000 + 32 sources) directly into **Qwen2.5-7B-Instruct**.
- **Target Model**: `Mossad-pentast-ai`
- **Source Weights**: Node `mmossad2324` (`bugbounty_copilot_lora`)
- **Outputs**:
  1. Standalone 16-bit Float16 Model (`/kaggle/working/Mossad-pentast-ai`)
  2. Standalone GGUF Q4_K_M Quantized Model (`/kaggle/working/Mossad-pentast-ai-GGUF`)


In [ ]:
# 1. Environment & CUDA Configuration
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers triton xformers


In [ ]:
# 2. Locate and Prepare LoRA Weights
import os, glob, json, urllib.request
LATEST_LORA_URLS = {"bugbounty_copilot_lora/README.md": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..p8DCNHYD_Za_TpNQ6zViow.u62AieKanJn3VCNvBuk_CGR3AYoCnVlXk7rUL7a-igE2zO9L7k3eOqZunLUIgfXoX_SrXFmccuFk983ut48pbxn3vqytwWYhQZ0ivI3BbzdPs2M_pqPvh3oRSjrBVAokHXPjFdcYCSWAJuM_0ayzHHFhu700o87xkrh8pxeAkLd3ghsqTg6aOlmw5kfRGCtLheXmZz-PB-o4hgjTqIZl09961bIU8hfeli7AbR0tgzV4TkMO_-li_x67vLX2tcLg1ieGYb2WzDFtnFMf91P2FithiiZX_NEP8aTTvmcOcXTRWNhNH1wWL9Wm8fgY6E9S_LXTUEcNdPZB0Tri0qZxVFwd3vn5aARNONHJ-Xz7FHnUeqNL3vEmOK3ZxU3gb8SWpEDt9-cbdjp_jVv0wxD-vrblMI52jo9aibutJHpbSCsU7-77NuInvvjUOcVLS_Ju00vLLkTvZ6JQAb6_Ofa0YkrfoF8_zGmlI7RhJkXI3HxDLjFPgp3CNE_2SCrAYlF9f8izlsf-VddKoGP6jNvkxzMo-IhvKT4iS5Q6F-msDS9EXmoDOT-hOdYcu2HXCA82nuxacQrHxMaK6pC6JIWA-Kq69SPFui9S6ZcXy-0Mxi3pkopQXlxCJAubf8dZrMmr.X_d0IX1unZSNpgNImk63NQ/bugbounty_copilot_lora/README.md", "bugbounty_copilot_lora/adapter_config.json": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..NgvH4vud19VU2CaJwu0w7g.ILDJKQDX1FGLIVluz0Ymr292uKb7QEouDrh5UuPo2uYFxeIJUMpFA9P8hInlJOIemB5gbGkwtCfoCP-fD1ZTkWRnuRzMRm28h9lOzkHI0z2Gw9UcT3Tu8erbxKhegxO98c97tRwtEcs_--WKDo1may8FnkqL-TKBgQRaqvc2El23GFllPMrM0x0q7J1GL-KCFpOIboj07cEQNCXcR1l8gxEtHDm34tDNGQPpOAZ750dS2hWn9vGu1uDOE13plGl26LFT5-sptJCMhHVRbn67UyAzNMkwwZpKTDaMwt6MMP1PMt6rf_W7C80lbks2IoXInIubXuNQgULfN6YvdUDdAjS695nNWPmRqEXYAUtyUcVZsXK-SERyZMrgLHGBARe1hHNKwjgSGHflpryn7jpZrK68MhyXALwacfe2sjqTRBwGZGiMPkDCBnPIODHWUstQl6H-oF6Z23GQ79tiFlepf4q07u7pDmnxZW9gIFrdVRe4uwL37LK1srPHweyK1iMjSvupBHl7Q6rvqjKpaSq-hhxaG_Qc6-v9XvpyEg6hL1xpZidWmo6el6FXT6cXAGLEaHZg3c8AG5bCJNnryvmKV2L56tOk6DFJpBO0nzddIqVCYAeLKb2eamqylEEq8cf3.6_ZP5z6sDOja5-OOOTFDYw/bugbounty_copilot_lora/adapter_config.json", "bugbounty_copilot_lora/adapter_model.safetensors": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..3vCuEV39SpHdpSMZmgkAyQ.uHehyexhZtWIl3KyasXzwKIGDUdOeds_2cCr_5OGULQWQyZNIrGraCl9y_yUNZQLxBcG3EgQvZX-ga8m4P5hE0qzbUcIjoEFeKgC7_xF0R0H4zYrc9A2RLcvwjCDYZQrmKthnf_NrJfFF4Q-vDtk3K7sK4UA_lCYS-ALvTEHXYsP-a4dhLss5DoShaXrVp_m085p0x4zX_RqlPQ2kbZ0E-p53C4KPJSmgTZ918yQfLnCg7Shc-ze5FqojZOxRe9wBiLK5CZWtUrCVReLnOb1FVP6nxctGg_X7JLAWNT8gU2NK6c5WsS9DuhNMTmx1NAZliMHaTdKGqrDrKxW9UHKBFJ_9TfPFKSIKYevqZXFiKkZaxzJlxCpfQ-jANHCotlKAHKAi3fA0i-hdtvEfilspEPIXVLO9YFddoa5UaWZAmu_JjzOKEmmQILD36LFj5v6OLfY_QP-vAX3MdZcW4RAwXuWt70pSnJfXj4IDyw9flpF83u4e2z3GyLd3hSwscStnAmToOyqyXQjpQJi0G8A0XQpjn-Qn__oPNUz3iDYpSp5X6QvbHz-SxuOxnnN_I5T6pmFw9lSUMACDu7rSh7ifJvlM51ruQk-J9WAdTC_Xsu572czS2uFXpMZ9br1L7u7.yEY0Wmz7wZQuaJbEhT6z8Q/bugbounty_copilot_lora/adapter_model.safetensors", "bugbounty_copilot_lora/chat_template.jinja": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..1ojGYGEwcrtV5WdibIE2ew.JDG_xIdxT4G4j7o_xzv6bn2PPzMSER1EtWazHsZiUvlHNRMZU1t7KQBXXM8qj9YWrNuwdfW3zElqBQnxe_Zz1p8LuTUWdqvC045UjN0jGpHALjkklCKRsHyXTOoHfktLHvO2GOkrE7neCjTDkRmDUcPWRoRXf64o2BPBcwu53XQLRKMGRa0BRqkoJ6WiOx3iwoTmAXmWGryOStfHhz0z44fRC4WA-aRnaEGQQQfC_KUIoYyRHdYGlM5I8cZV9wkRnau1FGDqohP1YsSShHA8KP3ItJRl-ThoOfSCQLmWzfyxjwVVW_rE73zsomfbPy9Sqb1L6wtQfUE9uZ0bElo40O92LrXx-8ljXBIR5F_kAO2TubvYhXKI80wX5kJJlKDlzEZB6AVoKMa0iNsUrE5-eKC5YJj0vIbaMCXKzQPUuGkAu8Ct39OO3xSBQTC8De062IsY1kux8g8MgP9VZs0T1IXTdCz_cXcpH_DsMutj7G0xGDrSq21pW6UnGCRTWIWVrBPlKHlV0mUwXioknUQlVq0TTyLjNIVdwY3BqlDZG_D-eQlmCbvJq7y0q2AfCDBRb-Z3qrWiKdiHhuKr8dASI16VYwMkelP-sl61GDmlK25oZXmhI4ZgtXFxDqloV6I3.8-92Ly1MiafIOUsdViaYBg/bugbounty_copilot_lora/chat_template.jinja", "bugbounty_copilot_lora/tokenizer.json": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..FCk8_dSAr6KJpt0-Q8d-Pg.DPv5t00ZmglpwQc41dWkFU5qJOlKdpN7EJ73nIroRJHjsiuiYJzi4bgqJciVVjn8yGSFvIIRjbG3Xh5FdAnjk3JJakpygCjUSVqpTXtmnP8wZstpKM-DUYgGhQ9NMwEu3uJJQcAWp0-ZzFf7UX4wtDAeR-VKyd6jOMpPgryZ8GuXFxmqmrd9REKOzB_sCV84eTE374Qd8-yf6oqJKDltnCVm7VHs8M4eQGArKi5mmd4RaU9lBZZpV-fqhs5KXOb6UaET3a5eYhtNAxZAo1a-7oczQoWGzdct8paKIgizd7ZqYJ17MdQpk3a8nceiagXUIIxySGXTTz7n_ZYRhua82DGgJSJ2OkJO5WR9W-TAOmASZpg0Tgr82VASgJHfXxdPZTTCyVVSphhdpYTj0LB2bjuFNK0HRKX-lPzA1iNkQyz1-F5btDRAyNXgSJgHwLqpMxox6G9JsstciAlCt7n4oTZk9tGqVMLUuO5CyQ1XG9SUY9al-SvuHFUqTZ95XJ4GFvcSjWV3pqxi5khx5wV6XGmNI3_Mq_l8XITc2L49kAzzyr-NqhSIN-PprBnbIp-n3vkbRNc_wS4A-y8KipNjH-EKBmvT3wbSdYYq84qIqR687A-zPLzevTAfDt5e1LdQ.vWks6yQ0nTEb_6i2zUnMUg/bugbounty_copilot_lora/tokenizer.json", "bugbounty_copilot_lora/tokenizer_config.json": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..3bpnXllik7Rm4yah9F0W1g.bi1Y2GGq9n9cdvyr6MZi79hs8Hh-rfwExB6QTjUa8WDTtr3aCKEBQXoZiv3wU4i9AgWOdCqv29Lk_wrdhw7cgTomLOfaI7h8pxq39OMncLMyRPJy9jod7aksuel2GPNWaNq2RgfTz_vrhJDtKAIdKot4Yi_4udtjhUUpTo1-dT2J00p-wbo65Df7aeQRZd9PH9vMGtezje18LUPoS9MQSKko3mlpk15AbnnxzO1NABcHWg-0Wn52rDThZyY-EjLySeOVuJY1ldjR0qmK8gnsUm3TKU-athIYb4pu_Bz_Ya01sGLkip8U6pH1vETjImL_m9QbCMRApT2LDLXziWXrlyHwO4Csrv9JUAW-m8xKG7wLVvxGcXhV_RIlFQBmf12SwdCRB2pEZxFs_xtQ4nO7RfFAsMR1Is0vdgBuyaStt_AqhL20lQiQgo8fxLNDizQRf6DK7ms3q_un6oE0aGA_mbA-uT7mksQArxQc32zbV9SFF1TAFj4Pxt9dzvFWiq96peLsnAOSeYo4XbOPOB750Nb7BXC9StYr0HcRHFOv8c6ojHrPvKcWAG1fbBRa57ZLpdJZ6nTB4an4Vo5pxpqQdDmHzRr_fJ2p5qgZSOy5l1eFgegkIBdgxyGl9eRDm7_F.dHmTWUzRdKPpSztTVHaW8w/bugbounty_copilot_lora/tokenizer_config.json"}

possible_loras = glob.glob('/kaggle/input/**/bugbounty_copilot_lora/adapter_model.safetensors', recursive=True)
if not possible_loras:
    possible_loras = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

adapter_dir = None
if possible_loras:
    adapter_dir = os.path.dirname(possible_loras[0])
    print(f'[✓] Found mounted LoRA Adapter at: {adapter_dir}')
elif LATEST_LORA_URLS:
    print(f'[*] Downloading {len(LATEST_LORA_URLS)} LoRA files via signed direct URLs...')
    adapter_dir = '/kaggle/working/latest_lora'
    os.makedirs(adapter_dir, exist_ok=True)
    for fname, url in LATEST_LORA_URLS.items():
        target_file = os.path.join(adapter_dir, os.path.basename(fname))
        try:
            urllib.request.urlretrieve(url, target_file)
            print(f'    -> Downloaded: {os.path.basename(fname)}')
        except Exception as e:
            print(f'    -> Warning: {e}')

if not adapter_dir or not os.path.exists(os.path.join(adapter_dir, 'adapter_model.safetensors')):
    raise RuntimeError('Could not locate trained LoRA weights for merging!')

print(f'[✓✓✓] LoRA weights verified ready at: {adapter_dir}')


In [ ]:
# 3. Load Model in 16-bit and Fuse LoRA Weights Losslessly
from unsloth import FastLanguageModel
import torch

print('[*] Loading base model and fusing LoRA adapter into full 16-bit neural tensors...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_dir,
    max_seq_length = 4096,
    dtype = torch.float16,
    load_in_4bit = False,
    device_map = {'': 0},
)
print('[✓] Model & LoRA weights successfully unified into memory graph!')


In [ ]:
# 4. Save Standalone 16-bit Model as 'Mossad-pentast-ai'
merged_dir = '/kaggle/working/Mossad-pentast-ai'
print(f'[*] Saving merged 16-bit standalone model to {merged_dir}...')
model.save_pretrained_merged(merged_dir, tokenizer, save_method = 'merged_16bit')
print(f'[✓✓✓] STANDALONE MODEL "Mossad-pentast-ai" SAVED SUCCESSFULLY!')


In [ ]:
# 5. Export Compact Standalone GGUF (Q4_K_M) for Fast Local / Server Inference
gguf_dir = '/kaggle/working/Mossad-pentast-ai-GGUF'
print(f'[*] Compiling and exporting GGUF (Q4_K_M) format to {gguf_dir}...')
try:
    model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method = 'q4_k_m')
    print(f'[✓✓✓] GGUF QUANTIZED MODEL "Mossad-pentast-ai" EXPORTED SUCCESSFULLY!')
except Exception as e:
    print(f'[!] GGUF compilation note ({e}). Standalone 16-bit model is 100% saved and ready.')


In [ ]:
# 6. Live Benchmark Inference Test on Merged Model 'Mossad-pentast-ai'
try:
    FastLanguageModel.for_inference(model)
    test_messages = [
        {"role": "system", "content": "You are Mossad-pentast-ai, an Elite Offensive Security & Bug Bounty AI Copilot."},
        {"role": "user", "content": "Target: https://api.enterprise.com/v1/auth/oauth/token. Detail your systematic exploit methodology to identify host header injection, token leakage, and unauthorized password takeover."}
    ]
    inputs = tokenizer.apply_chat_template(
        test_messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = 'pt',
    ).to('cuda:0')
    print('\n' + '='*70)
    print('🎯 LIVE BENCHMARK INFERENCE TEST: Mossad-pentast-ai')
    print('='*70)
    outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
    print(tokenizer.decode(outputs[0], skip_special_tokens = True))
    print('='*70)
    print('\n🎉 [FUSION COMPLETE] Mossad-pentast-ai IS FULLY BORN AND READY FOR OPERATION!')
except Exception as e:
    print(f'[!] Inference test note ({e}). Model is saved.')
